In [2]:
# STEP 1: Set month and year of interest for dashboard
year = "26"
month = "01"

### Stops inventory and metrics
This code will produce a dataframe containing the stops including the new University Hopsital and Doan Hall stops.

In [7]:
import pandas as pd
import numpy as np
import os
import pathlib

In [8]:
def build_stops_df():
    '''
    Build stops dataframe by merging static pattern stops and stop inventory files. Anytime that stops change, this will need to be re run after the csv files are updated.

    Parameters:
        None
    Returns:
        stops_df (pd.DataFrame): dataframe with stop id, stop name, and lat/lon coordinates for all stops in the pattern stops file
    '''
    # set directory paths for stop data - stored as 2 csv files in ./stops/
    stop_data_dir = pathlib.Path(os.getcwd()) / "stops"
    pattern_stops_path = stop_data_dir / "pattern_stops.csv"
    stop_inventory_path = stop_data_dir / "stop_inventory.csv"

    # read in static stop files
    pattern_stops = pd.read_csv(pattern_stops_path, header = None) # no header
    stop_inventory = pd.read_csv(stop_inventory_path)

    # only need cols 0 and 9 and can drop any dups
    pattern_stops = pattern_stops.iloc[:, [0, 9]].drop_duplicates()
    pattern_stops.columns = ["ROUTE", "STOP_ID"] # rename cols for merge

    # Merge pattern stops and stop inventory on stop id - left merge 
    stops_df = pattern_stops.merge(stop_inventory, how = "left", on = "STOP_ID")

    return stops_df

In [9]:
def which_stop(lat, lon, route = "MC", max_distance = 0.0005):
    '''
    Determine the closest stop to a given latitude and longitude. Med center route is default, but can be updated to other routes as needed. 
    NOTE: stops_df must be a global variable for this function to work, so build_stops_df() must be run before this function is called.

    Args:
        lat (float): The latitude of the point of interest.
        lon (float): The longitude of the point of interest.
        route (str): The route to filter stops by. Default is "MC" for medical center. Based on ROUTE col in stops DataFrame. Potentially important in future for overlapping stops.
        max_distance (float): The maximum distance to consider for a stop. Default is 0.0005 per legacy R code. approximately 150ish feet

    Returns:
        int: The STOP_ID of the closest stop.
    '''
    # subset the stops to route of interest
    route_stops = stops_df[stops_df['ROUTE'] == route]

    distance_lon = route_stops['LONG'].values - lon
    distance_lat = route_stops['LAT'].values - lat

    # calculate the distance to each stop
    distances = np.sqrt(distance_lon**2 + distance_lat**2)

    mask = distances < max_distance
    # lowest distance should be the closest stop now. - each stop is at minimum approx 430ft apart so margin of 150 should be fine for MC route.
    selected_stop = route_stops[mask]

    # if no stops within alloted distance, return None
    if not np.any(mask):
        return None

    return int(route_stops.loc[mask, 'STOP_ID'].iloc[0])


In [10]:
# NOTE: Will need to update function with year and month functionality.
def process_mc_busstate():
    """
    Process the busstate data for the medical center route.
    NOTE: This will ONLY work for the medical center route.
    NOTE: This will return a significantly smaller dataframe
    Args:
        None
    Returns:
        DataFrame: A pandas DataFrame containing the processed busstate data for the medical center route.
    """
    year_full = "2025" # add function to convert, etc
    month_full = "DEC"

    busstate_dir = pathlib.Path(os.getcwd()) / "BusState Cleaned"
    busstate_path = os.path.normpath(os.path.join(busstate_dir, f"{year_full}-{month_full}-busstate.csv"))
    busstate_df = pd.read_csv(busstate_path) # Now cleaned busstate data read in

    # Process the busstate data for medical center routes
    #busstate_df.info()

    # should filter to just MC routes
    filtered_busstate_df = busstate_df.loc[(busstate_df['RUN_ID'] > 1500) & (busstate_df['RUN_ID'] < 1600)].copy()

    # Convert time metrics to datetime
    filtered_busstate_df['EVENT_TIME'] = pd.to_datetime(filtered_busstate_df['EVENT_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['DEPARTURE_TIME'] = pd.to_datetime(filtered_busstate_df['DEPARTURE_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['ENTER_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['ENTER_STOP_WINDOW_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['EXIT_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['EXIT_STOP_WINDOW_TIME'], format = "%H:%M:%S")

    # sort by date and event time
    filtered_busstate_df = filtered_busstate_df.sort_values(['DATE', 'EVENT_TIME'])

    # Assign stop ID - most resource intensive step - as INT not float
    filtered_busstate_df['STOP_ID'] = filtered_busstate_df.apply(lambda row: which_stop(row['LATITUDE'], row['LONGITUDE']), axis = 1)

    # filter out 'dummy' stops, 27 and 461
    filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isin([27, 461])].index)

    # filter out empty stops - where bus was on route and not at a stop geographically
    filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isna()].index)

    # Convert to int
    filtered_busstate_df['STOP_ID'] = filtered_busstate_df['STOP_ID'].astype(int)

    # filter by bus id, date, event time
    filtered_busstate_df = filtered_busstate_df.sort_values(['BUS_ID', 'DATE', 'EVENT_TIME']).reset_index(drop = True) # restting index

    # create a new col for each new event
    # when stop changes OR bus_id changes OR elapsed time > 60
    count = (
        (filtered_busstate_df['STOP_ID'] != filtered_busstate_df['STOP_ID'].shift(1)) |
        (filtered_busstate_df['BUS_ID'] != filtered_busstate_df['BUS_ID'].shift(1)) |
        ((filtered_busstate_df['EVENT_TIME'] - filtered_busstate_df['EVENT_TIME'].shift(1)).dt.total_seconds() > 60)
    )

    # take cumsum of count to assign 
    filtered_busstate_df['COUNT'] = count.cumsum()

    # consolidate df
    consolidated_busstate_df = filtered_busstate_df.groupby(['BUS_ID', 'DATE', 'COUNT', 'STOP_ID', 'RUN_ID'], as_index = False).agg(
        BOARDINGS = ('BOARDINGS', 'sum'), # this WAS max
        ALIGHTINGS = ('ALIGHTINGS', 'sum'), # this WAS max
        LOAD = ('PASSENGER_LOAD', 'max'),
        EARLY_EVENT=("EVENT_TIME", "min"),
        LATE_EVENT=("EVENT_TIME", "max"),
        DEPARTURE_TIME=("DEPARTURE_TIME", "max"),
        ENTER_STOP=("ENTER_STOP_WINDOW_TIME", "min"),
        EXIT_STOP=("EXIT_STOP_WINDOW_TIME", "max"),
        RUN_ID = ("RUN_ID", "last"),
        DEST=("DEST_SIGN_ROUTE_TEXT", "last"),
    )
    #print(consolidated_busstate_df)

    # calculate the arrival and departure times for each event
    consolidated_busstate_df['ARRIVAL'] = consolidated_busstate_df[['EARLY_EVENT', 'ENTER_STOP']].min(axis=1) # arrival is earlier enter stop or early event
    consolidated_busstate_df['DEPARTURE'] = consolidated_busstate_df[['LATE_EVENT', 'DEPARTURE_TIME', 'EXIT_STOP']].max(axis=1) # departure is latest of late event or departure time or exit stop
    consolidated_busstate_df['DWELL'] = (consolidated_busstate_df['DEPARTURE'] - consolidated_busstate_df['ARRIVAL']) # total dwell time at stop

    # can drop unnecessary cols now
    consolidated_busstate_df = consolidated_busstate_df.drop(columns = ['EARLY_EVENT', 'LATE_EVENT', 'DEPARTURE_TIME', 'ENTER_STOP', 'EXIT_STOP'])

    # add hour and minute cols for filtering 
    consolidated_busstate_df['HOUR'] = consolidated_busstate_df['ARRIVAL'].dt.hour
    consolidated_busstate_df['MINUTE'] = consolidated_busstate_df['ARRIVAL'].dt.minute

    # ok - consolidated busstate df should now be ready for processing of the metrics.
    return consolidated_busstate_df

In [11]:
# This will produce df that metrics can be calculated from
stops_df = build_stops_df()

mc_busstate_consolidated = process_mc_busstate()

## Calculate the headways for each stop

In [12]:
mc_busstate_consolidated
mc_busstate_consolidated.to_csv("mc_busstate_consolidated.csv", index = False)

In [13]:
# mc_busstate_consolidated['DWELL']
mc_busstate_consolidated['DATE'] = pd.to_datetime(mc_busstate_consolidated['DATE'], format = "%Y-%m-%d")
mc_busstate_consolidated.info()

<class 'pandas.DataFrame'>
RangeIndex: 36989 entries, 0 to 36988
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype          
---  ------      --------------  -----          
 0   BUS_ID      36989 non-null  int64          
 1   DATE        36989 non-null  datetime64[us] 
 2   COUNT       36989 non-null  int64          
 3   STOP_ID     36989 non-null  int64          
 4   BOARDINGS   36989 non-null  int64          
 5   ALIGHTINGS  36989 non-null  int64          
 6   LOAD        36989 non-null  int64          
 7   RUN_ID      36989 non-null  float64        
 8   DEST        36989 non-null  str            
 9   ARRIVAL     36989 non-null  datetime64[us] 
 10  DEPARTURE   36989 non-null  datetime64[us] 
 11  DWELL       36989 non-null  timedelta64[us]
 12  HOUR        36989 non-null  int32          
 13  MINUTE      36989 non-null  int32          
dtypes: datetime64[us](3), float64(1), int32(2), int64(6), str(1), timedelta64[us](1)
memory usage: 3.7 MB


In [14]:
def calculate_headway(busstate_df, stop_id):
    '''
    Calculate the headway for a given stop id in the busstate dataframe.

    Args:
        busstate_df (pd.DataFrame): The busstate dataframe containing the bus events.
        stop_id (int): The stop ID for which to calculate headways.
    Returns:
        pd.DataFrame: A subset dataframe containing the headways for the specified stop ID.
    '''
    # calculate headways of carmack 2 stop - 403
    stop_hw = busstate_df[busstate_df['STOP_ID'] == stop_id]

    stop_hw = stop_hw.sort_values(['DATE', 'ARRIVAL'])

    # stop_hw['ARRIVAL'].isna().sum() # zero NA
    stop_hw['HEADWAY'] = stop_hw['ARRIVAL'] - stop_hw['ARRIVAL'].shift(1) # headway in minutes

    stop_hw = stop_hw.loc[
        (stop_hw['HEADWAY'].dt.total_seconds() / 60 >= 0) & # filter out negative headways
        (stop_hw['HEADWAY'].dt.total_seconds() / 60 < 22) # filter out headways greater than 22 minutes - per legacy R code
        ] 

    # adjust for midnight arrivals where hour == 0
    stop_hw['DATE'] = stop_hw['DATE'].where(stop_hw['ARRIVAL'].dt.hour != 0,
                                                        stop_hw['DATE'] - pd.Timedelta(days = 1))
    stop_hw['HOUR'] = stop_hw['ARRIVAL'].dt.hour.where(stop_hw['ARRIVAL'].dt.hour != 0, 24)

    stop_hw.groupby(['DATE', 'ARRIVAL'])
    
    return stop_hw

In [15]:
carmack_2_id = 403
carmack_3_id = 404
university_hospital_id = 401
doan_hall_id = 37

carmack_2_hw = calculate_headway(mc_busstate_consolidated, carmack_2_id)
carmack_3_hw = calculate_headway(mc_busstate_consolidated, carmack_3_id)
university_hospital_hw = calculate_headway(mc_busstate_consolidated, university_hospital_id)
doan_hall_hw = calculate_headway(mc_busstate_consolidated, doan_hall_id)

# Now have headways for each stop in separate dataframes, can calculate metrics from here.
# If we wanted to calculate headway metrics BY STOP, we could do that here without concatenating

In [16]:
# combine all headways into one df for metrics calculation
combined_hw = pd.concat([carmack_2_hw, carmack_3_hw, university_hospital_hw, doan_hall_hw], ignore_index = True)

(combined_hw['HEADWAY'].dt.total_seconds() / 60).mean() # should be the average headway across all 4 stops for month of Dec in minutes.
(combined_hw['HEADWAY'].dt.total_seconds() / 60).median() # should be the median headway across all 4 stops for month of Dec in minutes.

np.float64(3.45)

In [17]:
combined_hw['HEADWAY'].describe()

count                     26226
mean     0 days 00:04:27.012049
std      0 days 00:04:00.855841
min             0 days 00:00:00
25%             0 days 00:01:44
50%             0 days 00:03:27
75%             0 days 00:06:00
max             0 days 00:21:59
Name: HEADWAY, dtype: object

In [18]:
combined_hw.info()
combined_hw.isna().sum() # zero NA
combined_hw[combined_hw['HEADWAY'] == pd.Timedelta(seconds = 0)] # 73 total headways of 0 seconds
combined_hw[25400:25403] # 2 busses arrived at the same time at the same stop at doan - doesnt seem impossible?

<class 'pandas.DataFrame'>
RangeIndex: 26226 entries, 0 to 26225
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype          
---  ------      --------------  -----          
 0   BUS_ID      26226 non-null  int64          
 1   DATE        26226 non-null  datetime64[us] 
 2   COUNT       26226 non-null  int64          
 3   STOP_ID     26226 non-null  int64          
 4   BOARDINGS   26226 non-null  int64          
 5   ALIGHTINGS  26226 non-null  int64          
 6   LOAD        26226 non-null  int64          
 7   RUN_ID      26226 non-null  float64        
 8   DEST        26226 non-null  str            
 9   ARRIVAL     26226 non-null  datetime64[us] 
 10  DEPARTURE   26226 non-null  datetime64[us] 
 11  DWELL       26226 non-null  timedelta64[us]
 12  HOUR        26226 non-null  int32          
 13  MINUTE      26226 non-null  int32          
 14  HEADWAY     26226 non-null  timedelta64[us]
dtypes: datetime64[us](3), float64(1), int32(2), int64(6), str(1), ti

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,HEADWAY
25400,1904,2025-12-26,14215,37,0,0,3,1502.0,MC,1900-01-01 07:05:53,1900-01-01 07:07:09,0 days 00:01:16,7,5,0 days 00:04:21
25401,2501,2025-12-26,27782,37,0,3,0,1501.0,MC,1900-01-01 07:05:53,1900-01-01 07:06:45,0 days 00:00:52,7,5,0 days 00:00:00
25402,2401,2025-12-26,22144,37,0,3,0,1503.0,MC,1900-01-01 07:09:57,1900-01-01 07:10:54,0 days 00:00:57,7,9,0 days 00:04:04


In [19]:
def calculate_headway_dashboard_metrics(headway_df):
    '''
    Calculate headway metrics for each target hour, intended for external reporting use.
    NOTE: Target hour is stored as a constant within this function, so if target hours change, this function will need to be updated.

    Args:
        headway_df (pd.DataFrame): The combined headway dataframe containing headway information and target hours.
    Returns:
        pd.DataFrame: A summary dataframe containing the percentage of headways that met the target, as well as the 50th, 75th, and 90th percentile headway times for each target hour.
    '''
    # Target headways to calculate % on time for each timeframe
    TARGET_HEADWAYS = { # based on R code and report
        "5-6a": 10,
        "6-7a": 3,
        "7-8a": 3,
        "8a-2p": 10,
        "2-8p": 5,
        "8-10p": 10,
        "10p-12a": 5
    }
    # add in target headways to headway_df based on hour of arrival
    headway_df['TARGET'] = headway_df['HOUR'].apply(
        lambda hour: TARGET_HEADWAYS["5-6a"] if hour == 5 else (
            TARGET_HEADWAYS["6-7a"] if hour == 6 else (
                TARGET_HEADWAYS["7-8a"] if hour == 7 else (
                    TARGET_HEADWAYS["8a-2p"] if 8 <= hour < 14 else (
                        TARGET_HEADWAYS["2-8p"] if 14 <= hour < 20 else (
                            TARGET_HEADWAYS["8-10p"] if 20 <= hour < 22 else (
                                TARGET_HEADWAYS["10p-12a"] if 22 <= hour or hour == 0 else np.nan
                            )
                        )
                    )
                )
            )
        )
    )
    # create min of headway col to compare to target
    headway_df['HEADWAY_MIN'] = headway_df['HEADWAY'].dt.total_seconds() / 60

    # create met col for met headway target
    headway_df['MET'] = ((headway_df['TARGET'].notna()) & (headway_df['HEADWAY_MIN'] <= headway_df['TARGET'])).astype(int)
    # 1 if met, 0 if not met, only calculate if target is not NA

    # create target hour where each hour is broken into the target timeframes
    headway_df['TARGET_HOUR'] = headway_df['HOUR'].apply(
        lambda hour: "5-6a" if hour == 5 else (
            "6-7a" if hour == 6 else (
                "7-8a" if hour == 7 else (
                    "8a-2p" if 8 <= hour < 14 else (
                        "2-8p" if 14 <= hour < 20 else (
                            "8-10p" if 20 <= hour < 22 else (
                                "10p-12a" if 22 <= hour or hour == 0 else np.nan
                            )
                        )
                    )
                )
            )
        )
    )

    # combined_hw['HEADWAY_MIN'].head(20)
    # sort by chronological order
    target_order = list(TARGET_HEADWAYS.keys())
    headway_df['TARGET_HOUR'] = pd.Categorical(headway_df['TARGET_HOUR'], categories = target_order, ordered = True)

    # use HOUR col to create a metrics table with current combined data
    headway_summary = (
        headway_df
        .groupby('TARGET_HOUR') # grupby HOUR to have metrics by each individual hour, or TARGET_HOUR for the target timeframes
        .agg(
            MET = ("MET", "sum"),
            COUNT = ("HEADWAY", "size"),
            p50 = ("HEADWAY_MIN", lambda x: x.quantile(0.5)),
            p75 = ("HEADWAY_MIN", lambda x: x.quantile(0.75)),
            p90 = ("HEADWAY_MIN", lambda x: x.quantile(0.9)),
        )
    )
    # count here is LIKELY artificially inflated due to the 4th stop being added.
    headway_summary['MET'] = headway_summary['MET'] / headway_summary['COUNT'] # convert to percentage of headways that met target for each hour
    headway_summary.drop(columns = ['COUNT'], inplace = True)
    headway_summary.sort_values('TARGET_HOUR')

    return headway_summary
        

In [20]:
headway_summary = calculate_headway_dashboard_metrics(combined_hw)
headway_summary

,MET,p50,p75,p90
TARGET_HOUR,,,,
5-6a,0.959285,5.233333,6.850000,8.706667
6-7a,0.883029,1.183333,2.033333,3.250000
7-8a,0.721342,1.833333,3.250000,4.616667
8a-2p,0.908382,6.000000,8.116667,9.950000
2-8p,0.843833,2.933333,4.116667,5.583333
8-10p,0.963942,5.900000,7.337500,8.716667
10p-12a,0.511354,4.933333,6.483333,9.200000


In [21]:
def calculate_headway_internal_metrics(headway_df):
    '''
    Calculate internal headway metrics for each target hour, intended for internal use only and not external reporting. 
    This also can be configured to calculate metrics by stop if necessary, but currently is set up to calculate across all stops for each hour.
    This function is intended to be edited for future use as needed internally.

    Args:
        headway_df (pd.DataFrame): The combined headway dataframe containing headway information and target hours.
    Returns:
        pd.DataFrame: A summary dataframe containing headways by hour with summary statistcs includign mean, median, min, max, etc
    '''
    headway_summary = (
        headway_df
        .groupby('HOUR') # grupby HOUR to have metrics by each individual hour
        .agg(
            COUNT = ("HEADWAY", "size"),
            MEAN = ("HEADWAY_MIN", "mean"),
            MEDIAN = ("HEADWAY_MIN", "median"),
            MIN = ("HEADWAY_MIN", "min"),
            MAX = ("HEADWAY_MIN", "max"),
            p50 = ("HEADWAY_MIN", lambda x: x.quantile(0.5)),
            p75 = ("HEADWAY_MIN", lambda x: x.quantile(0.75)),
            p90 = ("HEADWAY_MIN", lambda x: x.quantile(0.9)),
        )
    )

    return headway_summary



In [22]:
headway_internal_summary = calculate_headway_internal_metrics(combined_hw)
headway_internal_summary
headway_internal_summary.to_csv("headway_internal_summary.csv")

## Calculate the capacity metric

In [23]:
# combine UH and Doan stops for capacity
# Carmack -> UH is inbound, Doan -> Carmack is outbound
# Treating UH and Doan as one stop.
def combine_uh_doan_stops(df, max_gap_minutes = 5):
    """
    Combine University Hospital and Doan stops into a single stop in the stop inventory for metrics.

    Args:
        stop_inventory (DataFrame): A DataFrame containing the stop inventory information.
    Returns:
        DataFrame: A DataFrame with University Hospital and Doan stops combined into a single stop with ID of 999.
    """
    df = df.copy()

    # will keep the original stop column for UH and Doan for reference, but assign a new combined stop - potential for debugging purposes
    # Assigning the number 999 for combined stop - no actual meaning for this and an unused stop number
    df['STOP_ORIGINAL'] = df['STOP_ID'] # keep original stop id for reference

    doan_stop_id = 37
    uh_stop_id = 401
 
    combined_id = 999 # completely abritary 

    max_gap = pd.Timedelta(minutes=max_gap_minutes)

    sort_cols = ['BUS_ID', 'DATE', 'RUN_ID', 'ARRIVAL', 'DEPARTURE']
    df = df.sort_values(sort_cols).reset_index(drop=True)

    combined_rows = []
    row_index = 0

    while row_index < len(df):
        current = df.iloc[row_index]

        if row_index < len(df) - 1:
            next_row = df.iloc[row_index + 1]

            # check if this is a UH->Doan pair within max_gap
            is_candidate_pair = (
                current['STOP_ID'] == uh_stop_id
                and next_row['STOP_ID'] == doan_stop_id
                and current['BUS_ID'] == next_row['BUS_ID']
                and current['DATE'] == next_row['DATE']
                and current['RUN_ID'] == next_row['RUN_ID']
                and pd.Timedelta(0) <= (next_row['ARRIVAL'] - current['DEPARTURE']) <= max_gap
            )

            if is_candidate_pair:
                merged = current.copy()
                merged['STOP_ID'] = combined_id
                merged['STOP_ORIGINAL'] = f"{uh_stop_id}_{doan_stop_id}"
                merged['BOARDINGS'] = current['BOARDINGS'] + next_row['BOARDINGS']
                merged['ALIGHTINGS'] = current['ALIGHTINGS'] + next_row['ALIGHTINGS']
                merged['LOAD'] = max(current['LOAD'], next_row['LOAD'])
                merged['DEPARTURE'] = next_row['DEPARTURE']
                merged['DWELL'] = merged['DEPARTURE'] - merged['ARRIVAL']
                merged['HOUR'] = merged['ARRIVAL'].hour
                merged['MINUTE'] = merged['ARRIVAL'].minute

                combined_rows.append(merged)
                row_index += 2  # skip next row because it was merged
                continue

        # if not merged, just add current
        combined_rows.append(current.copy())
        row_index += 1

    return pd.DataFrame(combined_rows).reset_index(drop=True)


#mc_rider = combine_uh_doan_stops(mc_busstate_consolidated)

In [33]:
def calculate_capacity_dashboard_metrics(busstate_df):
    '''
    Calculate capacity dashboard metrics for the given busstate dataframe.

    Args:
        busstate_df (pd.DataFrame): The busstate dataframe containing the bus events with combined UH and Doan stops.
    Returns:
        pd.DataFrame: A summary dataframe containing 
    '''
    # calculate the load based on max of boardings and alightings for each stop event
    mc_loads = busstate_df.copy()
    mc_loads['LOAD'] = mc_loads[['BOARDINGS', 'ALIGHTINGS']].max(axis=1)

    # NOTE: If an hourly view is desired, can edit this part
    # add time col with timeframes on the report
    mc_loads['TIME'] = mc_loads.apply(
        lambda row: 
            "5-6a" if row['HOUR'] <= 5 else
            "6-7a" if row['HOUR'] == 6 else
            "7-8a" if row['HOUR'] == 7 else
            "8a-2p" if 8 <= row['HOUR'] < 14 else
            "2-8p" if 14 <= row['HOUR'] < 20 else
            "8-10p" if 20 <= row['HOUR'] < 22 else
            "10p-12a" if row['HOUR'] >= 22 else np.nan,
        axis=1
    )

    # create size col based on load for each stop event - report metrics
    mc_loads['SIZE'] = mc_loads['LOAD'].apply(
        lambda load: "0-30 Passengers" if load <= 30 else
        "31-50 Passengers" if 30 < load <= 50 else
        "51-65 Passengers" if 50 < load <= 65 else
        "66+ Passengers" if load > 65 else None
    )

    print(mc_loads[mc_loads['SIZE'] == "66+ Passengers"]) 

    # create summary table for load size by time of day
    time_order = ['5-6a', '6-7a', '7-8a', '8a-2p', '2-8p', '8-10p', '10p-12a']

    # set time col as categorical with correct order for grouping and reporting
    mc_loads['TIME'] = pd.Categorical(
        mc_loads['TIME'],
        categories=time_order,
        ordered=True
    )

    # summarize the results
    load_summary = (
        mc_loads
        .groupby('TIME', observed=True)
        .agg(
            LOOPS=('SIZE', 'size'),
            p0_30=('SIZE', lambda x: (x == "0-30 Passengers").sum()),
            p31_50=('SIZE', lambda x: (x == "31-50 Passengers").sum()),
            p51_65=('SIZE', lambda x: (x == "51-65 Passengers").sum()),
            p66_plus=('SIZE', lambda x: (x == "66+ Passengers").sum()),
        )
        .reindex(time_order)
        .fillna(0)
        .reset_index()
        
        )
    return load_summary 


In [34]:
# # Compare boardings/alightings at combined stop vs separate stops to sanity check
# compare_cols = ['BOARDINGS', 'ALIGHTINGS', 'LOAD']

# print(mc_rider['STOP_ID'].value_counts())
# print(mc_busstate_consolidated['STOP_ID'].value_counts())

# print("================================")

# # print(mc_rider.info())
# # print(mc_busstate_consolidated.info())

# print(mc_rider['LOAD'].sum())
# print(mc_busstate_consolidated['LOAD'].sum())

# print("================================")

# print(mc_rider[compare_cols].sum())
# print(mc_busstate_consolidated[compare_cols].sum())
# mc_rider.head(20)
# mc_busstate_consolidated.head(20)

# Combine the UH and Doan stops to effectively treat them as one stop for capacity metrics
mc_rider = combine_uh_doan_stops(mc_busstate_consolidated)
mc_rider_combined_stop = mc_rider.loc[mc_rider['STOP_ID'] == 999].copy() 

capacity_summary = calculate_capacity_dashboard_metrics(mc_rider_combined_stop)
capacity_summary

       BUS_ID       DATE  COUNT  STOP_ID  BOARDINGS  ALIGHTINGS  LOAD  RUN_ID  \
27755    2503 2025-12-03  33105      999          8          68    68  1507.0   

      DEST             ARRIVAL           DEPARTURE           DWELL  HOUR  \
27755   MC 1900-01-01 08:48:48 1900-01-01 08:51:24 0 days 00:02:36     8   

       MINUTE STOP_ORIGINAL   TIME            SIZE  
27755      48        401_37  8a-2p  66+ Passengers  


,TIME,LOOPS,p0_30,p31_50,p51_65,p66_plus
0,5-6a,692,679,13,0,0
1,6-7a,651,580,70,1,0
2,7-8a,498,444,49,5,0
3,8a-2p,1168,1035,110,22,1
4,2-8p,2109,1845,254,10,0
5,8-10p,401,401,0,0,0
6,10p-12a,454,439,14,1,0


In [32]:
# Investigate the  66+ passengers load 
mc_rider['LOAD'].describe()
mc_rider[mc_rider['LOAD'] > 64]

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,STOP_ORIGINAL
22132,2501,2025-12-16,26362,95,6,0,65,1501.0,MC,1900-01-01 09:33:43,1900-01-01 09:34:39,0 days 00:00:56,9,33,95
22133,2501,2025-12-16,26363,999,11,59,65,1501.0,MC,1900-01-01 09:40:06,1900-01-01 09:43:15,0 days 00:03:09,9,40,401_37
29051,2503,2025-12-15,34640,37,36,0,65,1507.0,MC,1900-01-01 19:34:33,1900-01-01 19:34:56,0 days 00:00:23,19,34,37


## Travel Time Metric

In [27]:
def calculate_travel_time_dashboard_metrics(busstate_df):
    '''
    Calculate travel time dashboard metrics using the original Carmack 2 and Doan Hall stops.

    Args:
        busstate_df (pd.DataFrame): Original consolidated busstate data.

    Returns:
        pd.DataFrame: Travel time summary table with completed loop counts and average runtime by timeframe.
    '''
    time_order = ['5:30-7a', '7-10a', '10a-4p', '4-7p', '7p-12a', '12-5a']
    stop_labels = {403: 'CARMACK 2', 37: 'DOAN HALL'}
    valid_legs = ['CARMACK 2 - DOAN HALL', 'DOAN HALL - CARMACK 2']
    completed_loop_leg = 'DOAN HALL - CARMACK 2'

    def assign_ridecheck_time(hour_value):
        if 5.5 <= hour_value < 7:
            return '5:30-7a'
        if 7 <= hour_value < 10:
            return '7-10a'
        if 10 <= hour_value < 16:
            return '10a-4p'
        if 16 <= hour_value < 19:
            return '4-7p'
        if hour_value >= 19:
            return '7p-12a'
        return '12-5a'

    mc_rt = busstate_df.loc[busstate_df['STOP_ID'].isin([403, 37])].copy()
    mc_rt = mc_rt.sort_values(['BUS_ID', 'DATE', 'RUN_ID', 'ARRIVAL']).reset_index(drop=True)

    same_trip = (
        (mc_rt['BUS_ID'] == mc_rt['BUS_ID'].shift(1))
        & (mc_rt['DATE'] == mc_rt['DATE'].shift(1))
        & (mc_rt['RUN_ID'] == mc_rt['RUN_ID'].shift(1))
    )

    mc_rt['RUN_TIME'] = np.where(
        same_trip,
        (mc_rt['ARRIVAL'] - mc_rt['DEPARTURE'].shift(1)).dt.total_seconds() / 60,
        np.nan
    )
    mc_rt['PREV_STOP_ID'] = mc_rt['STOP_ID'].shift(1)
    mc_rt['PREV_STOP_LABEL'] = mc_rt['PREV_STOP_ID'].map(stop_labels)
    mc_rt['STOP_LABEL'] = mc_rt['STOP_ID'].map(stop_labels)
    mc_rt['LEG'] = np.where(
        same_trip,
        mc_rt['PREV_STOP_LABEL'] + ' - ' + mc_rt['STOP_LABEL'],
        np.nan
    )
    mc_rt['HOUR_FLOAT'] = mc_rt['HOUR'] + (mc_rt['MINUTE'] / 60)
    mc_rt['TIME'] = mc_rt['HOUR_FLOAT'].apply(assign_ridecheck_time)

    mc_rt = mc_rt.loc[mc_rt['LEG'].isin(valid_legs)].copy()
    mc_rt = mc_rt.loc[(mc_rt['RUN_TIME'] >= 0) & (mc_rt['RUN_TIME'] < 20)].copy()

    loop_summary = (
        mc_rt.loc[mc_rt['LEG'] == completed_loop_leg]
        .groupby('TIME', as_index=False)
        .agg(LOOPS=('LEG', 'size'))
    )

    runtime_summary = (
        mc_rt
        .groupby(['TIME', 'LEG'], as_index=False)['RUN_TIME']
        .mean()
    )
    runtime_summary['RUN_TIME'] = runtime_summary['RUN_TIME'].round(1)
    runtime_summary = runtime_summary.pivot(index='TIME', columns='LEG', values='RUN_TIME').reset_index()

    travel_time_summary = loop_summary.merge(runtime_summary, how='left', on='TIME')
    travel_time_summary['TIME'] = pd.Categorical(
        travel_time_summary['TIME'],
        categories=time_order,
        ordered=True
    )
    travel_time_summary = travel_time_summary.sort_values('TIME').reset_index(drop=True)

    return travel_time_summary

travel_time_summary = calculate_travel_time_dashboard_metrics(mc_busstate_consolidated)
travel_time_summary

,TIME,LOOPS,CARMACK 2 - DOAN HALL,DOAN HALL - CARMACK 2
0,5:30-7a,769,9.2,6.7
1,7-10a,982,11.9,6.9
2,10a-4p,1317,12.2,6.7
3,4-7p,1072,12.5,6.6
4,7p-12a,1232,11.9,5.9
5,12-5a,500,10.2,5.7


In [28]:
# Final sum up boardings
total_boardings = mc_busstate_consolidated['BOARDINGS'].sum()
print(f"{total_boardings} passengers")

100043 passengers


In [29]:
root_dir = pathlib.Path(os.getcwd()) / "Dashboard Data" / "2026" / "JAN"

# save to csv
headway_summary.to_csv(root_dir / f"1-Headway-{month}-{year}.csv", index = False)
capacity_summary.to_csv(root_dir / f"2-Capacity-{month}-{year}.csv", index = False)
travel_time_summary.to_csv(root_dir / f"3-TravelTime-{month}-{year}.csv", index = False)

NameError: name 'month' is not defined